In [1]:
# Imports
import qiskit
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

print("Qiskit Version:", qiskit.__version__)
simulator = AerSimulator()

def superdense_coding(message="00", bit_flip_error=False, shots=1024):
    if message not in {"00", "01", "10", "11"}:
        raise ValueError("Message must be 00, 01, 10, or 11")

    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)  # shared |Phi+> Bell pair
    qc.barrier()

    if message[0] == "1":
        qc.z(0)
    if message[1] == "1":
        qc.x(0)
    qc.barrier()

    if bit_flip_error:
        qc.x(0)

    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    counts = simulator.run(qc, shots=shots).result().get_counts()
    return qc, counts


def decoded_counts(counts):
    # Convert Qiskit displayed c1c0 keys to message order.
    return {key[::-1]: value for key, value in counts.items()}


def decoded_message(counts):
    return max(decoded_counts(counts), key=decoded_counts(counts).get)


Qiskit Version: 2.5.1


In [2]:
messages = ["00", "01", "10", "11"]
summary = []

for message in messages:
    _, counts = superdense_coding(message, shots=1024)
    decoded = decoded_message(counts)
    summary.append((message, decoded, decoded == message))
    print(f"Sent: {message} | Counts: {counts} | Decoded: {decoded} | Correct: {decoded == message}")

print("\nSummary")
print("Sent | Decoded | Correct")
for sent, decoded, correct in summary:
    print(f"{sent}   | {decoded}     | {correct}")


Sent: 00 | Counts: {'00': 1024} | Decoded: 00 | Correct: True
Sent: 01 | Counts: {'10': 1024} | Decoded: 01 | Correct: True
Sent: 10 | Counts: {'01': 1024} | Decoded: 10 | Correct: True
Sent: 11 | Counts: {'11': 1024} | Decoded: 11 | Correct: True

Summary
Sent | Decoded | Correct
00   | 00     | True
01   | 01     | True
10   | 10     | True
11   | 11     | True
